# PhyloGap: Cross-Species Genome Gap Reconstruction on Kaggle

This notebook clones the **PhyloGap** repository from GitHub, installs bioinformatics dependencies, executes the evaluation engine, runs baseline benchmark extractions, simulates autoregressive error decay, and validates the 4-tier B10K avian clade dataset.

**GitHub Repository:** [https://github.com/huanight19RaH/phylogap.git](https://github.com/huanight19RaH/phylogap.git)
**Environment:** Kaggle GPU (T4 x2 / P100) or CPU
**Internet Access:** Required (Enabled in Kernel Settings)

In [ ]:
# 1. Clone PhyloGap Repository from GitHub
import os
import sys
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/huanight19RaH/phylogap.git"
WORKSPACE_DIR = Path("/kaggle/working/phylogap")

if WORKSPACE_DIR.exists():
    print(f"Cleaning existing directory: {WORKSPACE_DIR}")
    shutil.rmtree(WORKSPACE_DIR)

print(f"Cloning PhyloGap from {REPO_URL}...")
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(WORKSPACE_DIR)], check=True)

os.chdir(WORKSPACE_DIR)
sys.path.insert(0, str(WORKSPACE_DIR))
sys.path.insert(0, str(WORKSPACE_DIR / "src"))
sys.path.insert(0, str(WORKSPACE_DIR / "scripts"))
os.environ["PYTHONPATH"] = f"{WORKSPACE_DIR}:{WORKSPACE_DIR / 'src'}:{WORKSPACE_DIR / 'scripts'}:" + os.environ.get("PYTHONPATH", "")

print(f"Successfully initialized in: {os.getcwd()}")

In [ ]:
# 2. Install Project Dependencies
!python -m pip install -q -r requirements.txt
print("Dependencies installed successfully.")

In [ ]:
# 3. Run Full Unit Test Suite (14 Tests)
print("Running PhyloGap unit test verification suite...")
!python -m unittest discover tests

In [ ]:
# 4. Run SOTA Baseline Benchmark Extraction (RQ1)
print("Extracting standardized SOTA baseline benchmarks on Drosophila and Calypte anna...")
!python scripts/benchmark_baseline_evaluator.py

In [ ]:
# 5. Run Autoregressive Nucleotide Error Decay Simulation (RQ2 / Gate 4)
print("Simulating error propagation and repetitive trapping across gap lengths...")
!python scripts/simulate_lm_gap_filling.py

In [ ]:
# 6. Query and Validate B10K 4-Tier Avian Clades Manifest (RQ3)
print("Querying NCBI Datasets API for curated avian clades...")
!python scripts/query_b10k_metadata.py

In [ ]:
# 7. Display Key Result Summaries
import pandas as pd

sota_csv = Path("experiments/results/baseline_benchmark_sota.csv")
if sota_csv.exists():
    print("=== SOTA Baseline Performance Table ===")
    df_sota = pd.read_csv(sota_csv)
    display(df_sota)

decay_csv = Path("experiments/results/autoregressive_decay_simulation.csv")
if decay_csv.exists():
    print("\n=== Autoregressive Error Decay Table ===")
    df_decay = pd.read_csv(decay_csv)
    display(df_decay)

manifest_csv = Path("data/manifests/avian_b10k_manifest.csv")
if manifest_csv.exists():
    print("\n=== Curated B10K Avian Manifest ===")
    df_manifest = pd.read_csv(manifest_csv)
    display(df_manifest[["tier", "role", "species", "accession", "assembly_name", "divergence_mya", "contig_n50", "scaffold_n50"]])

In [ ]:
# 8. Package Results into ZIP for Easy Download
import zipfile

archive_path = Path("/kaggle/working/phylogap_experiment_results.zip")
files_to_zip = list(Path("experiments/results").glob("*.csv")) + list(Path("data/manifests").glob("*.csv"))

with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in files_to_zip:
        zf.write(f, arcname=f.name)

print(f"Packed {len(files_to_zip)} result files into: {archive_path}")
print(f"File size: {archive_path.stat().st_size:,} bytes")